# Lesson 02 — Watershed Algorithm

## Why
Watershed separates touching objects that contour detection would merge into one.
Classic use case: overlapping coins, touching cells in microscopy.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))

# Distance transform: each pixel = distance to nearest background
dist_transform = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
_, sure_fg  = cv2.threshold(dist_transform, 0.5*dist_transform.max(), 255, 0)
sure_fg     = np.uint8(sure_fg)

# Sure background
sure_bg     = cv2.dilate(binary, se, iterations=5)
unknown     = cv2.subtract(sure_bg, sure_fg)

# Markers
_, markers  = cv2.connectedComponents(sure_fg)
markers     = markers + 1
markers[unknown==255] = 0

markers     = cv2.watershed(img, markers)
vis         = img.copy()
vis[markers==-1] = [0,0,255]  # watershed boundaries in red

fig, axes = plt.subplots(1, 4, figsize=(22,5))
for ax, im, t in zip(axes,
    [img, dist_transform, sure_fg, vis],
    ['Original', 'Distance Transform', 'Sure Foreground', 'Watershed Result']):
    ax.imshow(im if len(im.shape)==3 else im, cmap='gray' if len(im.shape)==2 else None)
    if len(im.shape)==3: ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(t); ax.axis('off')
plt.show()

## Key Takeaway
Watershed treats the image as a topographic surface. The distance transform creates the "hills".
Markers seed the regions. Watershed fills from seeds until regions meet — boundaries are marked red.